In [ ]:
import os
import pandas as pd

# 염색체 목록
chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y', 'M']

# 경로 설정
path_51_dir = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\annotateAll"
path_52_dir = r"E:\CAGI_data\dbNSFP5.1a_grch38_splits"
output_dir   = r"E:\CAGI_data\All_Missense_Annotated"
os.makedirs(output_dir, exist_ok=True)

# 불필요한 열
cols_to_drop_51 = [
    "hg19_chr", "hg19_pos(1-based)", "hg18_chr", "hg18_pos(1-based)",
    "genename", "cds_strand", "refcodon", "codonpos",
    "Ensembl_geneid", "Ensembl_transcriptid", "Ensembl_proteinid"
]

# 병합 키 생성 함수
def make_key(df, chr_col):
    return df[chr_col].astype(str) + "__" + \
           df["pos(1-based)"].astype(str) + "__" + \
           df["ref"].astype(str) + "__" + \
           df["alt"].astype(str) + "__" + \
           df["aaref"].astype(str) + "__" + \
           df["aaalt"].astype(str)

# 염색체별 처리
for chrom in chromosomes:
    print(f"🔄 Processing chr{chrom}...")

    # 1. 파일 경로
    file_51 = os.path.join(path_51_dir, f"dbNSFP5.1_nsSNV.chr{chrom}.gz")
    file_52 = os.path.join(path_52_dir, f"dbNSFP5.2a_{chrom}.tsv.gz")
    
    # 2. 데이터 불러오기
    df_51 = pd.read_csv(file_51, sep='\t', compression='gzip', low_memory=False, dtype=str)
    df_52 = pd.read_csv(file_52, sep='\t', compression='gzip', low_memory=False, dtype=str)

    # 3. 병합 키 생성
    df_51["__key__"] = make_key(df_51, "#chr")
    df_52["__key__"] = make_key(df_52, "chr")

    # 4. 5.2a에서 필요한 열만, aapos → mut_pos로 변경
    df_52_slim = df_52[["__key__", "Uniprot_acc", "Uniprot_entry", "aapos"]].copy()
    df_52_slim = df_52_slim.rename(columns={"aapos": "mut_pos"})
    df_52_slim = df_52_slim.drop_duplicates("__key__")

    # 5. 병합
    df_merged = df_51.merge(df_52_slim, on="__key__", how="left")

    # 6. 누락/중복 로그
    dups = df_52[df_52.duplicated("__key__", keep=False)]
    if not dups.empty:
        dups.to_csv(os.path.join(output_dir, f"log_duplicated_keys_chr{chrom}.tsv"), sep="\t", index=False)
    
    missing = df_merged[df_merged["Uniprot_acc"].isna()]
    if not missing.empty:
        missing.to_csv(os.path.join(output_dir, f"log_missing_uniprot_chr{chrom}.tsv"), sep="\t", index=False)

    # 7. 열 제거 및 저장
    df_merged.drop(columns=["__key__"], inplace=True)
    for col in cols_to_drop_51:
        if col in df_merged.columns:
            df_merged.drop(columns=col, inplace=True)

    out_path = os.path.join(output_dir, f"chr{chrom}.tsv")
    df_merged.to_csv(out_path, sep="\t", index=False)

    print(f"✅ chr{chrom} done. Saved to {out_path}")

print("\n🎉 All chromosomes processed!")


🔄 Processing chrX...
✅ chrX done. Saved to E:\CAGI_data\All_Missense_Annotated\chrX.tsv
🔄 Processing chrY...
✅ chrY done. Saved to E:\CAGI_data\All_Missense_Annotated\chrY.tsv
🔄 Processing chrM...
✅ chrM done. Saved to E:\CAGI_data\All_Missense_Annotated\chrM.tsv

🎉 All chromosomes processed!


In [9]:
df_merged

,#chr,pos(1-based),ref,alt,aaref,aaalt,aapos,Uniprot_acc,Uniprot_entry,mut_pos
0,M,3307,A,C,M,L,1,P03886,NU1M_HUMAN,1
1,M,3307,A,G,M,V,1,P03886,NU1M_HUMAN,1
2,M,3307,A,T,M,L,1,P03886,NU1M_HUMAN,1
3,M,3308,T,A,M,K,1,P03886,NU1M_HUMAN,1
4,M,3308,T,C,M,T,1,P03886,NU1M_HUMAN,1
...,...,...,...,...,...,...,...,...,...,...
25569,M,15884,G,C,A,P,380,P00156,CYB_HUMAN,380
25570,M,15884,G,T,A,S,380,P00156,CYB_HUMAN,380
25571,M,15885,C,A,A,D,380,P00156,CYB_HUMAN,380
25572,M,15885,C,G,A,G,380,P00156,CYB_HUMAN,380


In [1]:
import os
import pandas as pd

input_dir = r"E:\CAGI_data\All_Missense_Annotated"
output_tsv = r"E:\CAGI_data\All_Missense_Annotated\all_missense_variants_for_prediction.tsv"

dfs = []

for file in os.listdir(input_dir):

    print(file)
    
    if not file.endswith(".tsv") or file.startswith("log_"):
        continue

    path = os.path.join(input_dir, file)
    df = pd.read_csv(path, sep="\t", dtype=str)

    # 필수 컬럼 확인
    if not {"Uniprot_acc", "mut_pos", "aaref", "aaalt"}.issubset(df.columns):
        print(f"⚠️ Skipped {file}: required columns not found")
        continue

    # drop rows with missing split targets
    df = df.dropna(subset=["Uniprot_acc", "mut_pos", "aaref", "aaalt"])

    # split Uniprot_acc & mut_pos into exploded rows
    df["Uniprot_acc"] = df["Uniprot_acc"].str.split(";")
    df["mut_pos"] = df["mut_pos"].str.split(";")

    df = df.explode("Uniprot_acc").explode("mut_pos")

    df_clean = df[["Uniprot_acc", "mut_pos", "aaref", "aaalt"]].copy()
    df_clean.columns = ["UniProtID", "MutPos", "WT", "Mut"]

    dfs.append(df_clean)

# 모든 조각 합치고 중복 제거
df_out = pd.concat(dfs, ignore_index=True)
df_out.drop_duplicates(inplace=True)

# 저장
df_out.to_csv(output_tsv, sep="\t", index=False)
print(f"\n✅ Done! Output saved to:\n{output_tsv}")


chr1.tsv
chr2.tsv
chr3.tsv
chr4.tsv
chr5.tsv
chr6.tsv
chr7.tsv
chr8.tsv
chr9.tsv
chr10.tsv
chr11.tsv
chr12.tsv
chr13.tsv
chr14.tsv
chr15.tsv
chr16.tsv
chr17.tsv
chr18.tsv
chr19.tsv
chr20.tsv
chr21.tsv
chr22.tsv
chrX.tsv
chrY.tsv
chrM.tsv


MemoryError: Unable to allocate 57.3 GiB for an array with shape (4, 1924036937) and data type object

In [6]:
# 염색체 이름 순서 (dfs의 순서와 동일하다고 가정)
chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]
assert len(chroms) == len(dfs), "❗ dfs와 염색체 수 불일치!"

for df_chr, chrom in zip(dfs, chroms):
    out_path = os.path.join(input_dir, f"missense_for_prediction_{chrom}.tsv")
    df_chr.drop_duplicates(inplace=True)
    df_chr.to_csv(out_path, sep="\t", index=False)
    print(f"✅ Saved: {out_path}")

✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr1.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr2.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr3.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr4.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr5.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr6.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr7.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr8.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr9.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr10.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr11.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_prediction_chr12.tsv
✅ Saved: E:\CAGI_data\All_Missense_Annotated\missense_for_pre